In [6]:
import pandas as pd
import numpy as np



def encyclicEncoding(data):
    data=data.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)
    data["month_sin"] = np.sin(2 * np.pi * data["month_num"] / 12)
    data["month_cos"] = np.cos(2 * np.pi * data["month_num"] / 12)
    data["week_sin"] = np.sin(2 * np.pi * data["epi_week"] / 52)
    data["week_cos"] = np.cos(2 * np.pi * data["epi_week"] / 52)

    return data




In [2]:
import pandas as pd
import numpy as np




def handleDataInconsistency(data):
        
    data["district"] = data["district"].ffill()
    data["month"] = data["month"].ffill()

    data["cases"] = data["cases"].interpolate(method="linear")
    data["deaths"] = data["deaths"].interpolate(method="linear")


    data = data.drop_duplicates()

    data["district"] = data["district"].str.lower().str.strip()


    data["cases"] = data["cases"].clip(lower=0)
    data["deaths"] = data["deaths"].clip(lower=0)


    data = data[(data["epi_week"] >= 1) & (data["epi_week"] <= 52)]
    data = data[(data["month_num"] >= 1) & (data["month_num"] <= 12)]


    data = data.sort_values(["district", "year", "epi_week"])


    assert data["month_num"].between(1, 12).all()
    assert data["epi_week"].between(1, 52).all()

    return data


In [3]:
import numpy as np
import pandas as pd

def lagFeatures(df):

    df=df.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)

    df["t1_cases"]=df.groupby("district")["cases"].shift(1)
    df["t2_cases"]=df.groupby("district")["cases"].shift(2)
    
    df["T2M_lag1"]=df.groupby("district")["T2M_mean"].shift(1)

    df["PRECTOTCORR_lag1"]=df.groupby("district")["PRECTOTCORR_sum"].shift(1)
    df["PRECTOTCORR_lag2"]=df.groupby("district")["PRECTOTCORR_sum"].shift(2)

    df=df.dropna(subset=["t1_cases", "t2_cases"]).reset_index(drop=True)
    return df


In [4]:
def waterProxy(df):

    df=df.sort_values(by=["district","year", "epi_week"]).reset_index(drop=True)

    df["waterProxy"]=(df.groupby("district")["PRECTOTCORR_sum"]
                        .transform(lambda elem:elem.rolling(window=2,min_periods=1).sum()))
    
    return df


In [12]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
import joblib

df = pd.read_csv('../data/processed/punjab_district_weekly_merged.csv')

df = df.sort_values(["district", "year", "epi_week"]).reset_index(drop=True)

df = encyclicEncoding(df)
df = lagFeatures(df)
df = waterProxy(df)
df = handleDataInconsistency(df)

trainYears = [2016, 2017, 2018, 2019]
trainMask  = df['year'].isin(trainYears)

encoder = OrdinalEncoder()
encoder.fit(df[trainMask][['district']])
df['district_encoded'] = encoder.transform(df[['district']])
joblib.dump(encoder, "../models/district_encoder.pkl")

trainYearsThreshold = [2016, 2017, 2018]
trainMaskThreshold  = df['year'].isin(trainYearsThreshold)

districtThreshold = df[trainMaskThreshold].groupby('district')['cases'].quantile(0.90)
df['districtThreshold'] = df['district'].map(districtThreshold)
df['prevCases']  = df.groupby("district")["cases"].shift(1)
df['isOutbreak'] = (df['prevCases'] > df['districtThreshold']).astype(int)
df = df.drop(columns=['districtThreshold', 'prevCases'])

trainPool = df[df['year'].isin(trainYears)].copy()
test      = df[df['year'] == 2020].copy()


In [13]:
#Feature Engineering 
#in our case,the features are mostly the derived and engineered rows like the lag features,water proxy,encycling-features and the weather condition while our target is the no. of cases

features = [
    "district_encoded",
    "T2M_mean", "RH2M_mean", "PRECTOTCORR_sum", "WS10M_mean",
    "T2M_lag1", "PRECTOTCORR_lag1", "PRECTOTCORR_lag2",
    "t1_cases", "t2_cases",
    "waterProxy",
    "month_sin", "month_cos", "week_sin", "week_cos",
    "isOutbreak"
]
target = "cases"

xTrainPool = trainPool[features].copy()
yTrainPool = trainPool[target].copy()
xTest      = test[features].copy()
yTest      = test[target].copy()

trainPool.to_csv("../data/processed/trainPool.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

print(f"Train pool rows : {len(trainPool)} — 2016-2019")
print(f"Test rows       : {len(test)}      — 2020")
print(f"\nOutbreak weeks in train pool : {trainPool['isOutbreak'].sum()}")
print(f"Outbreak weeks in test       : {test['isOutbreak'].sum()}")

Train pool rows : 6840 — 2016-2019
Test rows       : 1728      — 2020

Outbreak weeks in train pool : 1103
Outbreak weeks in test       : 117


In [15]:
# Check before modeling
print(trainPool.isnull().sum())

year                0
month               0
month_num           0
epi_week            0
region              0
district            0
cases               0
deaths              0
T2M_mean            0
RH2M_mean           0
PRECTOTCORR_sum     0
WS10M_mean          0
month_sin           0
month_cos           0
week_sin            0
week_cos            0
t1_cases            0
t2_cases            0
T2M_lag1            0
PRECTOTCORR_lag1    0
PRECTOTCORR_lag2    0
waterProxy          0
district_encoded    0
isOutbreak          0
dtype: int64


In [ ]:
# #Scaling the data
# from sklearn.preprocessing import StandardScaler
# import joblib as jl

# scaler = StandardScaler()
# xTrainScaled = scaler.fit_transform(xTrain)

# xTestScaled = scaler.transform(xTest)
# jl.dump(scaler, "../models/scaler.pkl")

# print(f"Scaler means (first 5 features): {scaler.mean_[:5]}") 
# print(f"Scaler stds  (first 5 features): {scaler.scale_[:5]}")

Scaler means (first 5 features): [17.5        26.71619495 36.0387173   8.2737813   2.52306729]
Scaler stds  (first 5 features): [10.38829469  8.30447763 13.64367141 15.44411543  0.72919974]


In [ ]:

# severityMapping={
#     0:'Low',
#     1:'Medium',
#     2:'High',
#     3:'Critical'
# }

# q25=yTrain.quantile(0.25)
# q75=yTrain.quantile(0.75)
# q90=yTrain.quantile(0.90)

# def labelSeverityCases(cases):
#     if cases <= q25:
#         return 0  # Low
#     elif cases <= q75:
#         return 1  # Medium
#     elif cases <= q90:
#         return 2  # High
#     else:
#         return 3  # Critical
    

# yTrainSeverity = yTrain.apply(labelSeverityCases).values
                                              
# yTestSeverity = yTest.apply(labelSeverityCases).values


In [ ]:
# # freezing the data pipeline for saving the data from exposure to risks and corruption
# import numpy as np
# train.to_csv("../data/processed/train.csv", index=False)

# test.to_csv("../data/processed/test.csv", index=False)

# np.save("../data/processed/xTrainScaled.npy", xTrainScaled)
# np.save("../data/processed/ytrain_severity.npy", yTrainSeverity)